# Understanding the Roman WFI Bad Pixels Mask Reference File 

## Kernel Information and Read-Only Status

To run this notebook, please select "Roman Research Nexus {VERSION}" kernel at the top right of your window. For example "Roman Research Nexus 2026.1".

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.
    

## Introduction
The purpose of this notebook is to understand the the content and purpose of the **Bad Pixels Mask** (`MASK`) reference file.

More details about this and other reference files can be found in the [Reference File Information](https://roman-pipeline.readthedocs.io/en/stable/roman/references_general/index.html)

### Local Run Settings

If you want to run the notebook in your local machine, refer to the information in [local installation](../../markdown/local-run.md) instructions before proceeding with the notebook. The instructions provide inportant information about setting up your environment and installing dependnecies.

## Imports
Libraries used:
- *copy* for making copies of Python objects
- *crds* for access to calibration reference files
- *matplotlib* and *mpl_toolkits* for plotting images
- *numpy* for array manipulation
- *roman_datamodels* for opening Roman WFI ASDF files
- *asdf* for opening Roman WFI ASDF files
- *os* for operating system functions

In [ ]:
import os
import copy

import matplotlib.pyplot as plt
from matplotlib import colors, colormaps as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import roman_datamodels as rdm

### The Calibration Reference Data System (CRDS)

 The reference files, developed and validated by STScI’s Science Operations Center, are continually updated as new WFI data become available. For more information about how CRDS works and how it assigns the most appropriate reference file for each calibration step, refer to the notebook [Understanding CRDS and How to Select Calibration Reference files](crds_reference_files.ipynb). 

**IMPORTANT NOTE:** Reference files are a work in progress and will be updated several times before Roman launch. If you notice irregularities or missing information, please understand that they may be a known issue. If you have questions, please contact the [Roman Help Desk](https://romanhelp.stsci.edu).

In [ ]:
import crds

Now let's dive into this reference file.

### Bad Pixel Masks

Bad pixels are masked during the Data Quality (DQ) initialization step.  The MASK reference file, which contains static bad pixel flags (i.e., locations of dead pixels, telegraph pixels, etc.) for each detector, populates the data quality (DQ) `dq` array of the L2 calibrated rate image files after processing by RomanCal. During RomanCal processing, the MASK file is used in the `romancal.dq_init.DQInitStep()` step (the first step in the Exposure Pipeline) to populate an array called `pixeldq` in the *RampModel* datamodel, which is used within the pipeline. DQ flags are combined with additional DQ flags from reference file DQ arrays during subsequent processing steps using `bitwise_or`.

These mask reference files are created from dark or flat calibration datasets. Different types of pixels are flagged based on their pixel values (i.e., dead pixels with significantly reduced detector response) or behavior up-the-ramp (i.e., telegraph pixels jump between two electronic states).

For more details, see the [romancal documentation](https://roman-pipeline.readthedocs.io/en/latest/roman/dq_init/index.html) and [Rdox documentation](https://roman-docs.stsci.edu/data-handbook-home/roman-data-pipelines/exposure-level-pipeline#ExposureLevelPipeline-dq_init) for DQ initialization.

Let's check the environmental variables set for CRDS

In [ ]:
print(f"CRDS server location: {os.environ.get('CRDS_SERVER_URL')}")
print(f"CRDS context file: {os.environ.get('CRDS_CONTEXT')}")

If we want to change the context, we can do it in the next cell. In this case, we choose context `roman_0055.pmap`.

In [ ]:
os.environ['CRDS_CONTEXT']='roman_0055.pmap'

### Retrieving Reference Files

As you run the exposure pipeline, the most up-to-date reference files will be automatically selected for each step. However, if you would like to use a specific reference file, retrieve those from `crds` Python API and feed these to the Exposure Level or Mosaic Pipeline, see the notebook [Understanding CRDS and How to Select Calibration Reference files](crds_reference_files.ipynb) for more details. 

For the mask files in particular, the keywords that will identify the best reference file to use are:
- mask
    - ROMAN.META.INSTRUMENT.NAME
    - ROMAN.META.INSTRUMENT.DETECTOR
    - ROMAN.META.EXPOSURE.START_TIME
    
These keywords may be combined into a single dictionary to find and download the file using `crds.getreferences()`.  

In [ ]:
meta = {'ROMAN.META.INSTRUMENT.NAME': 'WFI',
        'ROMAN.META.INSTRUMENT.DETECTOR': 'WFI01',
        'ROMAN.META.EXPOSURE.START_TIME': '2026-01-01 00:00:00'
       }

ref_files = crds.getreferences(meta, reftypes=['mask'], observatory='roman')
ref_files

### Examining Reference Files

Reference files use `roman_datamodels` just like WFI science data products and can be accessed in the same way (see the tutorial [Working with ASDF](../working_with_asdf/working_with_asdf.ipynb) for more information). Let's take a closer look at the files we retrieved from our `crds.getreferences()` example starting with the mask file:

In [ ]:
mask = rdm.open(ref_files['mask'])
mask.info()

We see that the mask file contains metadata and a single array called `dq`. If we display the `dq` array, then we can see all of the features that have been flagged. The [Working with ASDF](../working_with_asdf/working_with_asdf.ipynb) tutorial gives more information about how to parse the meanings of the DQ flags. Let's get basic statistics for this MASK file.

In [ ]:
dq = mask.dq
print(f"DQ array shape: {dq.shape}")
print(f"Data type: {dq.dtype}")

# Unique DQ values
unique_dq = np.unique(dq)
print("\nUnique DQ values:", sorted(unique_dq))

# Overall statistics
total_pixels = dq.size
flagged_pixels = np.sum(dq > 0)
good_pixels = np.sum(dq == 0)

print(f"\nTotal pixels: {total_pixels:,}")
print(f"Good pixels (DQ=0): {good_pixels:,} ({good_pixels/total_pixels*100:.2f}%)")
print(f"Flagged pixels (DQ>0): {flagged_pixels:,} ({flagged_pixels/total_pixels*100:.2f}%)")

# 3. Breakdown by DQ value (only non-zero)
print("\nPixel count by DQ flag:")
for val in sorted(unique_dq):
    if val > 0:
        count = np.sum(dq == val)
        fraction = count / total_pixels * 100
        print(f"  DQ = {val:3d}: {count:10,} pixels ({fraction:6.3f}%)")

Putting this information in a different way

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(dq.flatten(), bins=len(unique_dq), edgecolor='black', log=True)
plt.xlabel('DQ Flag Value')
plt.ylabel('Number of Pixels (log scale)')
plt.title('Distribution of DQ Flags in Bad Pixel Mask')
plt.grid(True, alpha=0.3)

# Optional: annotate the most common flags
for val in unique_dq:
    if val > 0 and np.sum(dq == val) > total_pixels * 0.001:  # only label significant ones
        plt.text(val, np.sum(dq == val)*1.1, str(val), ha='center', fontsize=9)

plt.show()

Now, let's plot two versions of the file, one with each bitwise sum of DQ flags on a color map and another flattened version with "good" (DQ = 0; black pixels in the right-hand panel of the plot below) and "bad" (DQ >= 1; white pixels in the right-hand panel of the plot below) flags:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 14))

# Make a copy of the colormap so we can reset non-finite numbers to black
my_cmap = copy.copy(cm.get_cmap('nipy_spectral'))
my_cmap.set_bad((0, 0, 0))

# Display the mask file with a log normalization using the updated color map
im = axs[0].imshow(mask.dq, origin='lower', norm=colors.LogNorm(vmin=1, vmax=5e5), cmap=my_cmap)
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
axs[0].set_xlabel('Science X (pixels)')
axs[0].set_ylabel('Science Y (pixels)')
axs[0].set_title('Color-Coded DQ Flags')
fig.colorbar(im, cax=cax)

# Display the mask file with boolean good and bad values on a grayscale map
axs[1].imshow(np.bool(mask.dq), origin='lower', cmap='binary_r')
axs[1].set_xlabel('Science X (pixels)')
axs[1].set_ylabel('Science Y (pixels)')
axs[1].set_title('Good vs Bad Flags')

plt.tight_layout();

Note that not all DQ flags > 0 are necessarily bad. Many DQ flags are informational about detector effects or note something about the data processing. It is important for you to decide what effects are important for your science case.

## About this Notebook
**Author:** T. Desjardins. & R. Diaz

**Updated On:** 2026-07-06

<table width="100%" style="border:none; border-collapse:collapse;">

  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>